# Double Descent and Modern Generalization Phenomena

| Difficulty | ████████░░ 8/10 |
| :--- | :--- |
| Prerequisites | 13.01–13.05 |
| Optional | 11.03 (NTK), 11.04 (infinite-width networks) |
| Time | ~85 min |
| Colab GPU | Recommended |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/13_Statistical_Learning_Theory/08_double_descent.ipynb)

---

## 🎯 Learning Objective

Understand why massively overparameterized models generalize despite classical statistical
learning theory predicting they should not — through the lens of double descent, benign
overfitting, and implicit regularization. By the end of this notebook you will be able to
mathematically characterize the interpolation threshold, reproduce the double-descent curve
in both model-wise and epoch-wise settings, and state precise conditions under which
overfitting is benign.

---

## 📐 Theory

### 1. Classical U-curve vs. modern practice

The bias–variance decomposition (recall `13.02`) predicts that for a model family
parameterized by complexity $p$, the expected test error decomposes as

$$\mathbb{E}[\text{test error}] = \text{Bias}^2(p) + \text{Variance}(p) + \sigma^2_{\text{noise}}$$

where $\text{Bias}^2$ decreases and $\text{Variance}$ increases with $p$, yielding the
classical **U-shaped** curve with an optimal $p^\star$ at the minimum. This predicts that
models with $p \gg n$ (far more parameters than training samples) should have enormous
variance and fail catastrophically. Yet modern deep networks routinely operate with
$p/n \sim 10^3$–$10^6$ and *generalize well*. Something fundamental is missing from the
classical picture.

### 2. Double descent (Belkin et al., 2019)

The **double descent** phenomenon reconciles the classical and modern regimes. As $p$
increases through three distinct phases, the test error traces a characteristic curve:

**Phase I — Underparameterized ($p < n$).** The model cannot interpolate the training data.
The classical bias–variance tradeoff applies, and test error follows the U-shape. The model
minimizes the regularized empirical risk:

$$\hat{\theta} = \arg\min_{\theta \in \mathbb{R}^p} \frac{1}{n}\sum_{i=1}^n \ell(f_\theta(x_i), y_i)$$

and training error remains positive.

**Phase II — Interpolation threshold ($p \approx n$).** The model has *just enough*
capacity to perfectly fit the training data, including any label noise. The system
$\Phi \theta = y$ (where $\Phi \in \mathbb{R}^{n \times p}$ is the feature matrix) is
exactly determined (or nearly so), and the unique interpolating solution is *forced* to
pass through every noisy training point with no remaining degrees of freedom for
regularity. Formally, for the linear model $y = \Phi\theta + \varepsilon$ with
$\varepsilon \sim \mathcal{N}(0, \sigma^2 I_n)$ and $\text{rank}(\Phi) = n$, the
interpolating solution when $p = n$ is

$$\hat{\theta}_{\text{interp}} = \Phi^{-1} y$$

and the test error diverges as the condition number $\kappa(\Phi) \to \infty$. The
variance term scales as

$$\text{Variance} \propto \sigma^2 \cdot \text{tr}\bigl((\Phi^\top \Phi)^{-1}\bigr) \;\xrightarrow{p \to n}\; \infty$$

because $\Phi^\top \Phi$ becomes singular at $p = n$. This is the **interpolation peak**.

**Phase III — Overparameterized ($p \gg n$).** The system $\Phi\theta = y$ is
underdetermined: infinitely many solutions interpolate the data. Among these, gradient
descent (or equivalently, the pseudoinverse) selects the **minimum-norm interpolant**:

$$\hat{\theta}_{\text{min-norm}} = \Phi^\dagger y = \Phi^\top(\Phi\Phi^\top)^{-1} y$$

where $\Phi^\dagger$ is the Moore–Penrose pseudoinverse. This is the solution with smallest
$\|\theta\|_2$ among all interpolants. As $p \to \infty$, the prediction on a test point
$x_{\text{test}}$ becomes:

$$f(x_{\text{test}}) = x_{\text{test}}^\top \Phi^\top (\Phi\Phi^\top)^{-1} y$$

The noise component $\varepsilon$ gets spread across increasingly many directions in
$\mathbb{R}^p$, so $\|\hat{\theta}_{\text{min-norm}}\|_2 \to 0$ and the model's predictions
on *unseen* points become increasingly stable. Test error *decreases* — the second descent.

### 3. Model-wise double descent

**Model-wise double descent** occurs when we fix the dataset and increase the model's
width (or depth, or number of features). Let $p(w)$ denote the parameter count as a
function of width $w$. Plotting test error as a function of $w$ reveals the three-phase
curve described above. For a single hidden-layer network $f_w(x) = \frac{1}{\sqrt{w}}\sum_{j=1}^w a_j \sigma(b_j^\top x)$,
the interpolation threshold occurs at $w^\star$ where the number of learnable parameters
$p(w^\star) \approx n$.

### 4. Epoch-wise double descent (Nakkiran et al., 2020)

Even for a *fixed* model architecture, test error can exhibit double descent as a function
of **training time** $T$ (number of epochs). Early in training ($T$ small), the model
underfits; at an intermediate $T$, it memorizes the noise and reaches a local test-error
peak; with further training, gradient descent's implicit bias toward simple functions
takes over and test error decreases again. This is **epoch-wise double descent**.

The phenomenon is most pronounced when the model is slightly above the interpolation
threshold ($p \gtrsim n$) and the data contains label noise at rate $\eta > 0$. The
effective model complexity at epoch $T$ can be quantified as

$$C_{\text{eff}}(T) = \sum_{i=1}^{\min(p,n)} \frac{\lambda_i (1 - e^{-\eta_\text{lr} \lambda_i T})^2}{\lambda_i^2}$$

where $\lambda_i$ are eigenvalues of the empirical covariance $\Phi^\top \Phi / n$ and
$\eta_\text{lr}$ is the learning rate. At the critical $T^\star$ where
$C_{\text{eff}}(T^\star) \approx n$, test error peaks — mirroring the model-wise
interpolation threshold but in the time dimension.

### 5. Benign overfitting (Bartlett et al., 2020)

**Benign overfitting** formalizes when interpolation of *noisy* data is statistically
harmless. Consider the linear regression model $y = x^\top \theta^\star + \varepsilon$
with $x \sim \mathcal{N}(0, \Sigma)$ and $\varepsilon \sim \mathcal{N}(0, \sigma^2)$.
Let $\lambda_1 \ge \lambda_2 \ge \cdots \ge \lambda_p > 0$ be the eigenvalues of $\Sigma$.
Define the **effective rank** at level $k$:

$$r_k(\Sigma) = \frac{\text{tr}(\Sigma) - \sum_{i=1}^{k} \lambda_i}{\lambda_k} = \frac{\sum_{i=k+1}^{p} \lambda_i}{\lambda_k}$$

This measures how many "effective dimensions" exist beyond the top $k$ eigenvalues.

**Theorem (Bartlett et al., 2020).** The minimum-norm interpolant $\hat{\theta} = \Phi^\dagger y$
achieves near-optimal population risk if and only if the covariance spectrum decays
sufficiently fast. Specifically, benign overfitting occurs when:

$$r_k(\Sigma) \gg n \quad \text{for some } k = o(n)$$

Intuitively: the bulk of the spectrum (directions $k+1, k+2, \ldots, p$) must span a
very high-dimensional subspace (effective dimension $\gg n$), so the noise $\varepsilon$
gets diluted across many near-orthogonal directions and the minimum-norm interpolant's
excess risk satisfies:

$$\text{Excess Risk} = \mathbb{E}[\|\hat{\theta} - \theta^\star\|_\Sigma^2] \le \underbrace{\sigma^2 \frac{k}{n}}_{\text{signal subspace}} + \underbrace{\sigma^2 \frac{n}{r_k(\Sigma)}}_{\text{noise subspace}} + \underbrace{\sum_{i > k} \lambda_i (\theta^\star_i)^2}_{\text{approximation}}$$

When $r_k(\Sigma) \gg n$, the middle term vanishes and overfitting is benign. When
$r_k(\Sigma) \ll n$ (all eigenvalues comparable), noise cannot be diluted and overfitting
is **catastrophic** — the interpolation peak.

### 6. Reconciliation with bias–variance

The classical claim "more parameters $\Rightarrow$ more variance" is not a theorem — it
is an empirical observation that holds in the underparameterized regime but *breaks down*
in the overparameterized regime. For the minimum-norm interpolant, the variance on a
test point $x$ is:

$$\text{Var}[\hat{f}(x)] = \sigma^2 \, x^\top \Phi^\top (\Phi \Phi^\top)^{-2} \Phi \, x$$

As $p \to \infty$ with $\Phi \in \mathbb{R}^{n \times p}$, the matrix
$\Phi^\top(\Phi\Phi^\top)^{-2}\Phi$ has operator norm bounded by $1/\lambda_{\min}^2(\Phi\Phi^\top)$.
Under random feature models with $p/n \to \gamma > 1$, the Marchenko–Pastur law (recall `13.04`)
gives $\lambda_{\min}(\Phi\Phi^\top/p) \to (1 - 1/\sqrt{\gamma})^2 > 0$, so variance stays
bounded and can even *decrease* as $p$ grows. The minimum-norm interpolant is **stable**
to data perturbations precisely because the extra dimensions provide room to absorb noise.

### 7. Lottery ticket hypothesis (Frankle & Carlin, 2019)

The **lottery ticket hypothesis** provides a complementary perspective: a randomly
initialized dense network contains a sparse subnetwork (the "winning ticket") that, when
trained in isolation from the *same initialization*, matches the dense network's test
performance. Formally, for a network $f(x; \theta_0)$ with initial weights $\theta_0$,
there exists a binary mask $m \in \{0,1\}^p$ with $\|m\|_0 \ll p$ such that training
$f(x; m \odot \theta)$ from $m \odot \theta_0$ achieves comparable test error to training
the full $f(x; \theta)$ from $\theta_0$. This suggests overparameterization is needed
at *initialization* (to contain a good subnetwork with high probability) but not
necessarily at *convergence* — reconciling large $p$ at training time with low effective
complexity at test time.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
from numpy.linalg import pinv, norm, svd
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
rng = np.random.default_rng(42)

## 👁️ Visual Intuition

We reproduce the canonical double-descent curve using **Random Fourier Features** (RFF)
regression. We fix $n = 200$ training samples from a noisy sinusoidal target, then vary
the number of random features $p$ from $10$ to $2000$. For each $p$, we fit the
minimum-norm interpolant (when $p \ge n$) or the least-squares solution (when $p < n$)
and record train/test MSE.

**What to look for:** the U-shape for $p < n$, a sharp spike at $p \approx n$
(the interpolation threshold), and a second descent for $p \gg n$ where test error
*decreases* despite perfect training-set interpolation.

In [ ]:
# ── Double-descent curve with Random Fourier Features ──
n_train, n_test = 200, 500
sigma_noise = 0.3

# Ground-truth function
def true_fn(x):
    return np.sin(3 * x) + 0.5 * np.cos(7 * x)

X_train = rng.uniform(-np.pi, np.pi, n_train)
y_train = true_fn(X_train) + sigma_noise * rng.standard_normal(n_train)
X_test = rng.uniform(-np.pi, np.pi, n_test)
y_test = true_fn(X_test) + sigma_noise * rng.standard_normal(n_test)

# Random Fourier Features
omega = rng.standard_normal(2000)  # fixed random frequencies
b_rff = rng.uniform(0, 2 * np.pi, 2000)  # fixed random phases

def rff_features(X, p):
    """Map scalar X -> p-dim RFF feature vector."""
    return np.sqrt(2.0 / p) * np.cos(X[:, None] * omega[None, :p] + b_rff[None, :p])

p_values = np.unique(np.concatenate([
    np.arange(10, 180, 10),
    np.arange(180, 220, 2),   # fine grid near interpolation threshold
    np.arange(220, 500, 20),
    np.arange(500, 2001, 100)
]))

train_mse, test_mse = [], []
for p in p_values:
    Phi_tr = rff_features(X_train, p)
    Phi_te = rff_features(X_test, p)
    if p <= n_train:
        # Least-squares (possibly underdetermined at p = n)
        theta = pinv(Phi_tr) @ y_train
    else:
        # Minimum-norm interpolant via pseudoinverse
        theta = Phi_tr.T @ np.linalg.solve(Phi_tr @ Phi_tr.T, y_train)
    y_hat_tr = Phi_tr @ theta
    y_hat_te = Phi_te @ theta
    train_mse.append(np.mean((y_hat_tr - y_train)**2))
    test_mse.append(np.mean((y_hat_te - y_test)**2))

train_mse = np.array(train_mse)
test_mse = np.array(test_mse)

# ── Plot ──
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

# Left: full double descent curve
ax = axes[0]
ax.semilogy(p_values, train_mse, '-', color='#55A868', lw=2, label='Train MSE')
ax.semilogy(p_values, test_mse, '-', color='#C44E52', lw=2, label='Test MSE')
ax.axvline(n_train, color='#8172B2', ls='--', lw=1.5, label=f'p = n = {n_train}')
ax.set_xlabel('Number of features p', fontsize=12)
ax.set_ylabel('MSE (log scale)', fontsize=12)
ax.set_title('Double Descent: Random Fourier Features Regression', fontsize=13)
ax.legend(fontsize=10)
ax.set_xlim(0, 2000)

# Right: phase diagram schematic
ax = axes[1]
ratio = p_values / n_train
ax.semilogy(ratio, test_mse, '-', color='#4C72B0', lw=2)
ax.axvline(1.0, color='#8172B2', ls='--', lw=1.5, label='p/n = 1 (threshold)')
ax.fill_betweenx([1e-3, 1e3], 0, 1, alpha=0.08, color='#4C72B0', label='Underparameterized')
ax.fill_betweenx([1e-3, 1e3], 1, 10, alpha=0.08, color='#C44E52', label='Overparameterized')
ax.set_xlabel('Overparameterization ratio p/n', fontsize=12)
ax.set_ylabel('Test MSE (log scale)', fontsize=12)
ax.set_title('Phase Diagram: Test Error vs p/n', fontsize=13)
ax.legend(fontsize=10)
ax.set_xlim(0, 10)

plt.tight_layout()
plt.show()

# Numerical summary
peak_idx = np.argmax(test_mse)
best_overp = np.argmin(test_mse[p_values > n_train]) + np.searchsorted(p_values, n_train + 1)
print(f"Interpolation peak: p = {p_values[peak_idx]}, test MSE = {test_mse[peak_idx]:.4f}")
print(f"Best overparameterized: p = {p_values[best_overp]}, test MSE = {test_mse[best_overp]:.4f}")
print(f"Noise floor (σ² = {sigma_noise**2}): {sigma_noise**2:.4f}")
print(f"\nThe second descent brings test error BELOW the classical optimum — ")
print(f"overparameterization HELPS, contradicting the naive bias-variance prediction.")

## 🐍 Implementation from Scratch

We implement two key components from the theory:

1. **Minimum-norm interpolant** via the pseudoinverse — demonstrating the exact formula
   $\hat{\theta} = \Phi^\top(\Phi\Phi^\top)^{-1}y$ and verifying it achieves zero
   training error with the smallest $\|\theta\|_2$ among all interpolants.

2. **Effective rank** computation $r_k(\Sigma)$ and its connection to benign overfitting —
   we construct covariance matrices with different spectral decays (polynomial vs.
   exponential) and check the benign overfitting condition $r_k(\Sigma) \gg n$.

In [ ]:
# ── Part 1: Minimum-norm interpolant from scratch ──
print("=" * 70)
print("PART 1: Minimum-norm interpolant via pseudoinverse")
print("=" * 70)

n, p = 50, 500  # heavily overparameterized
Phi = rng.standard_normal((n, p)) / np.sqrt(p)
theta_true = rng.standard_normal(p) * 0.1
sigma = 0.2
y = Phi @ theta_true + sigma * rng.standard_normal(n)

# Method 1: via the formula θ = Φᵀ(ΦΦᵀ)⁻¹y
G = Phi @ Phi.T  # n × n Gram matrix
theta_mn = Phi.T @ np.linalg.solve(G, y)

# Method 2: via numpy pseudoinverse (should agree)
theta_pinv = pinv(Phi) @ y

# Verify
train_resid = norm(Phi @ theta_mn - y)
print(f"Training residual ||Φθ - y|| = {train_resid:.2e}  (should be ~0 = interpolation)")
print(f"||θ_formula - θ_pinv|| = {norm(theta_mn - theta_pinv):.2e}  (should be ~0)")
print(f"||θ_min_norm||₂ = {norm(theta_mn):.4f}")

# Compare with a RANDOM interpolant (not minimum-norm)
# θ_rand = θ_mn + null-space component
U, S, Vt = svd(Phi, full_matrices=True)
null_basis = Vt[n:]  # (p - n) × p
random_null = rng.standard_normal(p - n)
random_null /= norm(random_null)
theta_rand = theta_mn + 0.5 * null_basis.T @ random_null  # add null-space direction

print(f"\nRandom interpolant ||θ_rand||₂ = {norm(theta_rand):.4f}  (larger!)")
print(f"Random interpolant training residual = {norm(Phi @ theta_rand - y):.2e}  (still ~0)")
print(f"→ Both interpolate, but minimum-norm solution has SMALLER ||θ||₂")

# ── Part 2: Effective rank and benign overfitting conditions ──
print(f"\n{'=' * 70}")
print("PART 2: Effective rank and benign overfitting")
print("=" * 70)

def effective_rank(eigenvalues, k):
    """r_k(Σ) = (tr(Σ) - Σ_{i=1}^k λ_i) / λ_k"""
    total = np.sum(eigenvalues)
    top_k = np.sum(eigenvalues[:k])
    return (total - top_k) / eigenvalues[k - 1]

p_dim = 2000
n_samples = 200

# Spectrum 1: Polynomial decay λ_i ∝ i^{-α}
for alpha, label in [(0.5, 'slow (α=0.5)'), (1.0, 'moderate (α=1.0)'), (2.0, 'fast (α=2.0)')]:
    eigs = np.arange(1, p_dim + 1, dtype=float) ** (-alpha)
    k = 10
    rk = effective_rank(eigs, k)
    condition = 'BENIGN (r_k >> n)' if rk > 5 * n_samples else 'CATASTROPHIC (r_k ~ n)'
    print(f"Polynomial decay {label:20s}: r_{k}(Σ) = {rk:10.1f}, n = {n_samples} → {condition}")

# Spectrum 2: Exponential decay λ_i ∝ exp(-β·i)
for beta, label in [(0.001, 'slow (β=0.001)'), (0.01, 'moderate (β=0.01)'), (0.1, 'fast (β=0.1)')]:
    eigs = np.exp(-beta * np.arange(1, p_dim + 1))
    k = 10
    rk = effective_rank(eigs, k)
    condition = 'BENIGN (r_k >> n)' if rk > 5 * n_samples else 'CATASTROPHIC (r_k ~ n)'
    print(f"Exponential decay {label:20s}: r_{k}(Σ) = {rk:10.1f}, n = {n_samples} → {condition}")

print(f"\nKey insight: Benign overfitting requires r_k(Σ) >> n = {n_samples}.")
print("Slow spectral decay (many directions with non-negligible variance) dilutes noise.")
print("Fast spectral decay concentrates data in few dimensions → catastrophic overfitting.")

## 🤖 Why This Matters for AI

GPT-4 has $\sim 10^{12}$ parameters trained on $\sim 10^{13}$ tokens and generalizes
remarkably well to unseen text. Classical statistical learning theory (recall `13.01`)
predicts that a model with $p \gg n$ should *overfit catastrophically* — its VC dimension
or Rademacher complexity would suggest a vacuous generalization bound. Double descent and
benign overfitting are the theoretical reconciliation:

1. **Implicit regularization**: SGD on overparameterized networks converges to
   minimum-norm (or minimum-curvature) solutions, not arbitrary interpolants.

2. **High-dimensional dilution**: In modern architectures, the effective rank of the
   feature covariance is enormous — noise gets diluted across millions of near-orthogonal
   directions, exactly the benign overfitting condition.

3. **Scaling laws**: The second descent region predicts that *making models larger* can
   *reduce* test error even past the interpolation threshold — the theoretical basis for
   the scaling laws observed empirically in LLMs (next notebook: `13.09`).

Below we demonstrate epoch-wise double descent on an MLP trained on a noisy subset of
MNIST-like data, showing the phenomenon occurs not just in theory but in practical
neural network training.

In [ ]:
# ── Epoch-wise double descent on an MLP with noisy labels ──
# We use a synthetic classification task (no external data needed)

# Generate synthetic 2-class data in moderate dimension
d = 50
n_train_mlp, n_test_mlp = 150, 1000
noise_rate = 0.2  # 20% label noise

# True decision boundary: sign of first coordinate
X_tr = rng.standard_normal((n_train_mlp, d))
y_tr_clean = (X_tr[:, 0] > 0).astype(float)
# Flip labels with probability noise_rate
flip_mask = rng.random(n_train_mlp) < noise_rate
y_tr = y_tr_clean.copy()
y_tr[flip_mask] = 1 - y_tr[flip_mask]

X_te = rng.standard_normal((n_test_mlp, d))
y_te = (X_te[:, 0] > 0).astype(float)

# Simple 2-layer MLP (NumPy, no frameworks)
hidden = 300  # overparameterized: 300*50 + 300 + 300 + 1 = 15901 >> 150
lr = 0.01

def relu(z):
    return np.maximum(0, z)

def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

# Xavier initialization
W1 = rng.standard_normal((d, hidden)) * np.sqrt(2 / d)
b1 = np.zeros(hidden)
W2 = rng.standard_normal((hidden, 1)) * np.sqrt(2 / hidden)
b2 = np.zeros(1)

def forward(X, W1, b1, W2, b2):
    h = relu(X @ W1 + b1)
    logits = (h @ W2 + b2).ravel()
    return logits, h

def bce_loss(logits, y):
    p = sigmoid(logits)
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def accuracy(logits, y):
    return np.mean((sigmoid(logits) > 0.5) == y)

# Training loop — record metrics every few epochs
n_epochs = 2000
record_every = 10
history = {'epoch': [], 'train_loss': [], 'test_loss': [], 'train_acc': [], 'test_acc': []}

for epoch in range(1, n_epochs + 1):
    # Forward
    logits_tr, h_tr = forward(X_tr, W1, b1, W2, b2)
    p_tr = sigmoid(logits_tr)
    p_tr = np.clip(p_tr, 1e-7, 1 - 1e-7)
    
    # Backward (manual gradients)
    dlogits = (p_tr - y_tr) / n_train_mlp  # (n,)
    dW2 = h_tr.T @ dlogits[:, None]  # (hidden, 1)
    db2 = np.sum(dlogits)
    dh = dlogits[:, None] * W2.T  # (n, hidden)
    dh[h_tr <= 0] = 0  # ReLU backward
    dW1 = X_tr.T @ dh  # (d, hidden)
    db1 = np.sum(dh, axis=0)
    
    # Update
    W1 -= lr * dW1
    b1 -= lr * db1
    W2 -= lr * dW2
    b2 -= lr * db2
    
    if epoch % record_every == 0 or epoch == 1:
        logits_tr2, _ = forward(X_tr, W1, b1, W2, b2)
        logits_te, _ = forward(X_te, W1, b1, W2, b2)
        history['epoch'].append(epoch)
        history['train_loss'].append(bce_loss(logits_tr2, y_tr))
        history['test_loss'].append(bce_loss(logits_te, y_te))
        history['train_acc'].append(accuracy(logits_tr2, y_tr))
        history['test_acc'].append(accuracy(logits_te, y_te))

# ── Plot epoch-wise behavior ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(history['epoch'], history['train_loss'], color='#55A868', lw=2, label='Train loss')
ax.plot(history['epoch'], history['test_loss'], color='#C44E52', lw=2, label='Test loss')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('BCE Loss', fontsize=12)
ax.set_title('Epoch-wise Double Descent: Loss', fontsize=13)
ax.legend(fontsize=10)

ax = axes[1]
ax.plot(history['epoch'], history['train_acc'], color='#55A868', lw=2, label='Train acc')
ax.plot(history['epoch'], history['test_acc'], color='#4C72B0', lw=2, label='Test acc')
ax.axhline(1 - noise_rate, color='#8172B2', ls=':', lw=1.5,
           label=f'Clean Bayes rate ({1-noise_rate:.0%})')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Epoch-wise Double Descent: Accuracy', fontsize=13)
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

# Summary
peak_test_loss_idx = np.argmax(history['test_loss'])
final_test_acc = history['test_acc'][-1]
print(f"Model: 2-layer MLP, {d}→{hidden}→1 = {d*hidden + hidden + hidden + 1} params (n={n_train_mlp})")
print(f"Label noise rate: {noise_rate:.0%}")
print(f"Peak test loss at epoch {history['epoch'][peak_test_loss_idx]} ")
print(f"  (loss = {history['test_loss'][peak_test_loss_idx]:.4f})")
print(f"Final test accuracy: {final_test_acc:.4f}")
print(f"Train accuracy reaches 100% (interpolation) — yet test accuracy RECOVERS.")
print(f"This is epoch-wise double descent: continued training past interpolation HELPS.")

## 🏋️ Exercises

### 🔢 COMPUTE
1. Consider a linear system $\Phi \theta = y$ with $\Phi \in \mathbb{R}^{3 \times 5}$
   given by
   $\Phi = \begin{pmatrix} 1 & 0 & 1 & 0 & 0 \\ 0 & 1 & 0 & 1 & 0 \\ 0 & 0 & 1 & 0 & 1 \end{pmatrix}$
   and $y = (1, 2, 3)^\top$. Compute the minimum-norm interpolant
   $\hat{\theta} = \Phi^\top(\Phi\Phi^\top)^{-1}y$ by hand. Verify $\Phi\hat{\theta} = y$
   and that $\|\hat{\theta}\|_2$ is smaller than any other solution.

<details>
<summary>💡 Solution</summary>

First compute $\Phi\Phi^\top = \begin{pmatrix} 2 & 0 & 1 \\ 0 & 2 & 0 \\ 1 & 0 & 2 \end{pmatrix}$.
Its inverse is $\frac{1}{6}\begin{pmatrix} 4 & 0 & -2 \\ 0 & 3 & 0 \\ -2 & 0 & 4 \end{pmatrix}$.
Then $(\Phi\Phi^\top)^{-1}y = \frac{1}{6}(4 - 6, 6, -2 + 12)^\top = \frac{1}{6}(-2, 6, 10)^\top$.
Finally $\hat{\theta} = \Phi^\top \cdot \frac{1}{6}(-2, 6, 10)^\top = \frac{1}{6}(-2, 6, 8, 6, 10)^\top$.
Verify: $\Phi\hat{\theta} = \frac{1}{6}(-2+8, 6+6, 8+10)^\top = (1, 2, 3)^\top = y$ ✓.
$\|\hat{\theta}\|_2 = \frac{1}{6}\sqrt{4 + 36 + 64 + 36 + 100} = \frac{\sqrt{240}}{6} \approx 2.582$.
Any solution of the form $\hat{\theta} + v$ with $v$ in the null space of $\Phi$ also interpolates,
but $\|\hat{\theta} + v\|_2^2 = \|\hat{\theta}\|_2^2 + \|v\|_2^2 > \|\hat{\theta}\|_2^2$ since
$\hat{\theta} \perp \text{null}(\Phi)$ by construction.

</details>

### 💭 UNDERSTAND
2. Explain intuitively why the interpolation peak occurs at $p \approx n$ rather than at
   some other relationship between $p$ and $n$. Why does the test error *diverge* there
   but not at, say, $p = n/2$ or $p = 2n$?

<details>
<summary>💡 Solution</summary>

At $p < n$, the system $\Phi\theta = y$ is overdetermined — the model cannot interpolate and
instead finds a regularized (least-squares) solution that averages out noise. At $p > n$, the
system is underdetermined — infinitely many solutions interpolate, and the minimum-norm
solution spreads the "effort" across all $p$ dimensions, diluting noise. At exactly $p = n$,
the system is square and (generically) has a *unique* solution. This solution is forced to
pass through every noisy data point with zero remaining degrees of freedom — there is no
averaging (as in $p < n$) and no dilution (as in $p > n$). The condition number
$\kappa(\Phi)$ tends to infinity as the smallest singular value of $\Phi$ approaches zero
at $p \to n$, making the solution extremely sensitive to noise in $y$. This is why the
interpolation peak is specifically at $p \approx n$: it is the unique point where the model
has *just enough* capacity to memorize but *no room* to be stable.

</details>

### ✏️ DERIVE
3. For the overparameterized linear model with $p > n$ and feature matrix
   $\Phi \in \mathbb{R}^{n \times p}$ of rank $n$, derive that the minimum-norm
   interpolant is $\hat{\theta} = \Phi^\top(\Phi\Phi^\top)^{-1}y$ by solving the
   constrained optimization problem
   $\min_{\theta} \|\theta\|_2^2 \;\text{subject to}\; \Phi\theta = y$
   using the method of Lagrange multipliers.

<details>
<summary>💡 Solution</summary>

The Lagrangian is $\mathcal{L}(\theta, \lambda) = \|\theta\|_2^2 + \lambda^\top(\Phi\theta - y)$
where $\lambda \in \mathbb{R}^n$. Setting $\nabla_\theta \mathcal{L} = 0$:
$2\theta + \Phi^\top \lambda = 0 \implies \theta = -\frac{1}{2}\Phi^\top \lambda$.
Substituting into the constraint: $\Phi(-\frac{1}{2}\Phi^\top\lambda) = y \implies
-\frac{1}{2}\Phi\Phi^\top\lambda = y \implies \lambda = -2(\Phi\Phi^\top)^{-1}y$
(the inverse exists since $\text{rank}(\Phi) = n$). Back-substituting:
$\theta = -\frac{1}{2}\Phi^\top(-2(\Phi\Phi^\top)^{-1}y) = \Phi^\top(\Phi\Phi^\top)^{-1}y$.
This is precisely the Moore–Penrose pseudoinverse $\Phi^\dagger y$ when $\Phi$ has full
row rank. The second-order condition is trivially satisfied since the objective
$\|\theta\|_2^2$ is strictly convex. $\square$

</details>

### 🐍 IMPLEMENT
4. Reproduce the double-descent curve from the Visual Intuition cell, but add
   **ridge regression** with regularization $\lambda = 0.01$ in the underparameterized
   regime ($\hat{\theta} = (\Phi^\top\Phi + \lambda I)^{-1}\Phi^\top y$) and
   regularized minimum-norm in the overparameterized regime
   ($\hat{\theta} = \Phi^\top(\Phi\Phi^\top + \lambda I)^{-1}y$). Overlay both
   curves. Does ridge regularization eliminate the interpolation peak?

<details>
<summary>💡 Solution</summary>

```python
lam = 0.01
test_mse_ridge = []
for p in p_values:
    Phi_tr = rff_features(X_train, p)
    Phi_te = rff_features(X_test, p)
    if p <= n_train:
        theta = np.linalg.solve(Phi_tr.T @ Phi_tr + lam * np.eye(p), Phi_tr.T @ y_train)
    else:
        theta = Phi_tr.T @ np.linalg.solve(Phi_tr @ Phi_tr.T + lam * np.eye(n_train), y_train)
    test_mse_ridge.append(np.mean((Phi_te @ theta - y_test)**2))
```

Yes — ridge regularization smooths out the interpolation peak entirely. The regularized
curve monotonically decreases (or has a mild, non-divergent hump). This is because
$\lambda I$ prevents $(\Phi\Phi^\top)^{-1}$ from blowing up at the interpolation
threshold, bounding the condition number at $\le 1/\lambda$. The overparameterized
regime still benefits from many features, but the peak is gone.

</details>

### 🤖 APPLY
5. Modify the epoch-wise double descent MLP experiment in Cell 8: add $L_2$ weight
   decay ($\lambda = 10^{-4}$) to the gradient updates. Does weight decay suppress
   the epoch-wise double descent phenomenon? Plot both curves (with/without weight
   decay) and explain the result using the theory of implicit vs. explicit regularization.

<details>
<summary>💡 Solution</summary>

Add `dW1 += wd * W1` and `dW2 += wd * W2` with `wd = 1e-4` before the parameter updates.
Weight decay acts as explicit $L_2$ regularization, preventing the model from fully
interpolating the noisy labels. The training loss no longer reaches zero, so the model
never enters the "forced interpolation" regime that triggers the epoch-wise peak.
The test loss curve becomes monotonically decreasing (or with a very mild bump).
This illustrates that epoch-wise double descent is driven by the transition through
interpolation — explicit regularization that prevents interpolation suppresses the
phenomenon, at the potential cost of underfitting if $\lambda$ is too large.

</details>

### 🚀 CHALLENGE
6. **Benign overfitting critical-$\alpha$ experiment.** Generate data from
   $y = x^\top\theta^\star + \varepsilon$ with $\Sigma$ having polynomial spectral
   decay $\lambda_i = i^{-\alpha}$ for $i = 1, \ldots, p$. Fix $n = 100$, $p = 2000$,
   $\sigma = 0.5$, and vary $\alpha \in \{0.2, 0.5, 0.8, 1.0, 1.5, 2.0\}$. For each
   $\alpha$, compute the minimum-norm interpolant and its test error (averaged over 20
   trials). Plot test error vs. $\alpha$ and identify the critical $\alpha^\star$ where
   overfitting transitions from benign to catastrophic. Relate your finding to the
   effective rank condition $r_k(\Sigma) \gg n$.

<details>
<summary>💡 Solution</summary>

```python
n, p, sigma = 100, 2000, 0.5
alphas = [0.2, 0.5, 0.8, 1.0, 1.5, 2.0]
n_trials = 20
results = []
for alpha in alphas:
    eigs = np.arange(1, p+1, dtype=float)**(-alpha)
    Sigma_sqrt = np.diag(np.sqrt(eigs))
    theta_star = rng.standard_normal(p) * 0.1
    test_errors = []
    for _ in range(n_trials):
        Z = rng.standard_normal((n, p))
        X = Z @ Sigma_sqrt
        y = X @ theta_star + sigma * rng.standard_normal(n)
        theta_hat = X.T @ np.linalg.solve(X @ X.T, y)
        Z_te = rng.standard_normal((500, p))
        X_te = Z_te @ Sigma_sqrt
        y_te = X_te @ theta_star
        test_errors.append(np.mean((X_te @ theta_hat - y_te)**2))
    results.append(np.mean(test_errors))
```

You should observe a phase transition: for small $\alpha$ (slow decay, e.g., $\alpha = 0.2, 0.5$),
$r_k(\Sigma) \gg n$ and test error is low (benign overfitting). For large $\alpha$ (fast decay,
e.g., $\alpha = 1.5, 2.0$), the effective rank drops below $n$ and test error explodes
(catastrophic overfitting). The critical $\alpha^\star$ is typically around $0.8$–$1.0$
for these parameters — the boundary where $\sum_{i>k} i^{-\alpha}$ transitions from
divergent (benign) to convergent (catastrophic) as $p \to \infty$.

</details>

---

## 📚 Further Reading
- Belkin, Hsu, Ma & Mandal, ["Reconciling modern machine-learning practice and the classical bias–variance trade-off"](https://arxiv.org/abs/1812.11118) (PNAS 2019)
- Nakkiran, Kaplun, Bansal et al., ["Deep Double Descent"](https://arxiv.org/abs/1912.02292) (ICLR 2020)
- Bartlett, Long, Lugosi & Tsigler, ["Benign overfitting in linear regression"](https://arxiv.org/abs/1906.11300) (PNAS 2020)
- Hastie, Montanari, Rosset & Tibshirani, ["Surprises in High-Dimensional Ridgeless Least Squares Interpolation"](https://arxiv.org/abs/1903.08560) (Annals of Statistics 2022)
- Frankle & Carlin, ["The Lottery Ticket Hypothesis"](https://arxiv.org/abs/1803.03635) (ICLR 2019)

---

*Next notebook: `13.09` Scaling Laws and Compute-Optimal Training*